# Hybrid system model → website

This notebook defines the hybrid QC–HPC system shown in the 3D diagram on the homepage
(section *"From connected resources to system-wide value"*), runs a small workload simulation on it,
and exports the result to `web/public/data/hybrid-system.json`.

The website reads that file: node positions, which connections exist, which are active, and how much
traffic each connection carries (shown as moving pulses). Hovering a resource in the browser shows the metrics computed here.

**To change the diagram:** edit the layout or the simulation below and re-run the notebook.
On every push to `main`, GitHub Actions re-runs this notebook before building the site.

In [ ]:
# Parameters (papermill can override these)
output_path = "../web/public/data/hybrid-system.json"
seed = 7
n_steps = 2000          # simulated time steps
arrival_rate = 0.55     # mean workloads arriving per step
job_mix = {"classical": 0.5, "quantum": 0.3, "hybrid": 0.2}

In [ ]:
import json
from collections import deque
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd

rng = np.random.default_rng(seed)

## Layout

Coordinates are in plate units: the plate is 10 × 7, centred on (0, 0). `x` runs along the long edge,
`z` along the short edge. They were taken from the original SVG illustration so the 3D version lines up with it.

- **groups**: the floating tiles (HPC, Workloads, QC).
- **nodes**: `resource` nodes belong to a group; `hzero` nodes form the operational-integration layer on the plate.
- **links**: `step` controls the order in which connections appear when the diagram scrolls into view.

In [ ]:
plate = {"width": 10.0, "depth": 6.96, "thickness": 0.24, "radius": 0.22}

groups = [
    {"id": "hpc",       "label": "HPC",       "x": -2.98, "z":  1.73, "size": 2.0, "height": 0.86},
    {"id": "workloads", "label": "Workloads", "x":  0.06, "z": -0.44, "size": 2.0, "height": 0.86},
    {"id": "qc",        "label": "QC",        "x":  3.10, "z": -2.61, "size": 2.0, "height": 0.86},
]

# (index, layer, group, x, z, active, capacity)
node_rows = [
    (0,  "resource", "hpc",       -3.264,  2.176, True,  4),
    (1,  "resource", "hpc",       -2.393,  1.741, False, 4),
    (2,  "resource", "workloads", -0.653, -0.435, True,  None),
    (3,  "resource", "workloads", -0.218,  0.000, False, None),
    (4,  "resource", "workloads",  0.653, -0.435, True,  None),
    (5,  "resource", "qc",         2.829, -2.611, True,  1),
    (6,  "resource", "qc",         3.699, -3.046, False, 1),
    (7,  "resource", "qc",         3.264, -2.176, True,  1),
    (8,  "hzero",    None,        -3.916, -2.175, True,  None),
    (9,  "hzero",    None,        -3.046, -1.740, False, None),
    (10, "hzero",    None,        -0.871,  2.176, True,  None),
    (11, "hzero",    None,         0.435,  1.740, True,  None),
    (12, "hzero",    None,         2.611,  0.435, True,  None),
    (13, "hzero",    None,         3.481,  0.000, True,  None),
    (14, "hzero",    None,         3.046,  1.740, False, None),
    (15, "hzero",    None,         3.916,  2.175, True,  None),
]
nodes = pd.DataFrame(node_rows, columns=["index", "layer", "group", "x", "z", "active", "capacity"]).set_index("index")

# (from, to, step, active)
link_rows = [
    (0, 10, 3, True), (2, 11, 3, True), (4, 12, 3, True), (5, 8, 3, True), (7, 13, 3, True),
    (8, 10, 4, True), (10, 11, 4, True), (11, 12, 4, True), (12, 13, 4, True), (12, 15, 4, True),
    (1, 10, 5, False), (3, 14, 5, False), (6, 13, 5, False),
]
links = pd.DataFrame(link_rows, columns=["from", "to", "step", "active"])
nodes

## Workload simulation

Workloads are submitted from the active *Workloads* nodes. hzero routes each one over the integration layer
to the least-loaded resource that can run it (hybrid workloads use both an HPC and a QC resource).
We record utilisation and queues per resource, and how many workloads cross each connection.

In [ ]:
active_links = links[links.active]
adjacency = {i: [] for i in nodes.index}
for a, b in zip(active_links["from"], active_links["to"]):
    adjacency[a].append(b)
    adjacency[b].append(a)

def shortest_path(src, dst):
    prev, queue = {src: None}, deque([src])
    while queue:
        n = queue.popleft()
        if n == dst:
            break
        for m in adjacency[n]:
            if m not in prev:
                prev[m] = n
                queue.append(m)
    path, n = [], dst
    while n is not None:
        path.append(n)
        n = prev[n]
    return path[::-1]

sources = list(nodes[(nodes.group == "workloads") & nodes.active].index)
targets = {
    "classical": list(nodes[(nodes.group == "hpc") & nodes.active].index),
    "quantum":   list(nodes[(nodes.group == "qc") & nodes.active].index),
}
service_mean = {"classical": 6.0, "quantum": 3.0}

running = {n: [] for n in nodes.index}      # remaining service time of running jobs
queued = {n: deque() for n in nodes.index}  # (arrival_step, service_time)
busy_steps = {n: 0.0 for n in nodes.index}
queue_sum = {n: 0 for n in nodes.index}
done = {n: 0 for n in nodes.index}
waits = []
link_jobs = {tuple(sorted(k)): 0 for k in zip(links["from"], links["to"])}
routed = {n: 0 for n in nodes.index}
submitted = {n: 0 for n in nodes.index}

def load(n):
    return (len(running[n]) + len(queued[n])) / nodes.at[n, "capacity"]

def dispatch(kind, src, step):
    target = min(targets[kind], key=load)
    path = shortest_path(src, target)
    for a, b in zip(path, path[1:]):
        link_jobs[tuple(sorted((a, b)))] += 1
    for n in path[1:-1]:
        routed[n] += 1
    queued[target].append((step, rng.exponential(service_mean[kind])))

kinds, probs = list(job_mix), list(job_mix.values())
for step in range(n_steps):
    for _ in range(rng.poisson(arrival_rate)):
        kind, src = rng.choice(kinds, p=probs), rng.choice(sources)
        submitted[src] += 1
        for part in (["classical", "quantum"] if kind == "hybrid" else [kind]):
            dispatch(part, src, step)
    for n, cap in nodes.capacity.dropna().items():
        running[n] = [t - 1 for t in running[n] if t > 1]
        while queued[n] and len(running[n]) < cap:
            arrived, service = queued[n].popleft()
            waits.append(step - arrived)
            running[n].append(service)
            done[n] += 1
        busy_steps[n] += len(running[n]) / cap
        queue_sum[n] += len(queued[n])

print(f"Simulated {n_steps} steps, {len(waits)} workloads started, mean wait {np.mean(waits):.2f} steps")

## Results

In [ ]:
def r(v, d=1):
    return round(float(v), d)

node_metrics = {}
for n, row in nodes.iterrows():
    if not row.active:
        node_metrics[n] = {"status": "standby"}
    elif row.layer == "hzero":
        node_metrics[n] = {"status": "active", "workloads routed": int(routed[n])}
    elif row.group == "workloads":
        node_metrics[n] = {"status": "active", "workloads submitted": int(submitted[n])}
    else:
        node_metrics[n] = {
            "status": "active",
            "utilisation %": r(100 * busy_steps[n] / n_steps),
            "avg queue": r(queue_sum[n] / n_steps, 2),
            "workloads run": int(done[n]),
        }

max_jobs = max(link_jobs.values()) or 1
links["jobs"] = [link_jobs[tuple(sorted((a, b)))] for a, b in zip(links["from"], links["to"])]
links["flow"] = (links["jobs"] / max_jobs).round(3)

def group_metrics(gid):
    members = nodes[(nodes.group == gid) & nodes.active].index
    m = {"active nodes": f"{len(members)} / {int((nodes.group == gid).sum())}"}
    if gid == "workloads":
        m["workloads submitted"] = int(sum(submitted[n] for n in members))
    else:
        m["utilisation %"] = r(np.mean([100 * busy_steps[n] / n_steps for n in members]))
        m["workloads run"] = int(sum(done[n] for n in members))
    return m

summary = {
    "simulated steps": n_steps,
    "workloads started": len(waits),
    "mean wait (steps)": r(np.mean(waits), 2),
    "throughput / step": r(len(waits) / n_steps, 3),
}
display(pd.DataFrame(node_metrics).T)
links

In [ ]:
payload = {
    "generated_at": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    "source": "notebooks/hybrid_system.ipynb",
    "summary": summary,
    "plate": plate,
    "groups": [{**g, "metrics": group_metrics(g["id"])} for g in groups],
    "nodes": [
        {
            "id": f"n{n:02d}", "layer": row.layer, "group": None if pd.isna(row.group) else row.group,
            "x": row.x, "z": row.z, "active": bool(row.active), "metrics": node_metrics[n],
        }
        for n, row in nodes.iterrows()
    ],
    "links": [
        {"from": f"n{a:02d}", "to": f"n{b:02d}", "step": int(s), "active": bool(act), "flow": float(f), "jobs": int(j)}
        for a, b, s, act, f, j in links[["from", "to", "step", "active", "flow", "jobs"]].itertuples(index=False)
    ],
}
out = Path(output_path)
out.parent.mkdir(parents=True, exist_ok=True)
out.write_text(json.dumps(payload, indent=1, allow_nan=False))
print(f"Wrote {out.resolve()}")